# AIDA 2154 Final Project – 02 Segmentation

**Data version 2** (`shared/synthetic-data/kit-cards/`), generated from cleaned requests version 1.1.

**Question:** Can a simple, transparent pipeline find each object on a card as one region with a well-fitting box, providing suitable regions for the baseline classifier in notebook 03?

The pipeline uses classical methods covered in the course: denoising, local paper-colour estimation, colour or grayscale differences, thresholding, morphology, and connected-component detection. Predicted regions are matched to the recorded object boxes to assess detection and box fit. Missed objects, split objects, merged regions, and false alarms are reported separately.

**Use of the splits.** Training cards are used to inspect failures and develop a small settings grid. The grid and selection rule are fixed before comparing settings on the 120 validation cards; one later change, a band left out along the card edges, is described in section C. The selected setting is then frozen. Test images are not opened, measured, or used for development in this notebook; only their label records are loaded and then filtered out. The complete frozen detection-and-classification pipeline will be evaluated on test once, at the end of the project.

All cards are computer-generated and contain no personal information.

In [1]:
import hashlib
import json
from itertools import product
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
KIT = SD / "kit-cards"
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

ITEMS = ["access_badge", "safety_vest", "hard_hat", "laptop", "phone"]
DEV_SPLITS = ["train", "validation"]  # only development images are read; metadata includes all splits

# Only truly empty cells become missing (NaN); text such as "NA" stays as written
manifest = pd.read_csv(KIT / "manifest.csv", keep_default_na=False, na_values=[""])
objects = pd.read_csv(KIT / "icon_instances.csv", keep_default_na=False, na_values=[""])
generation = json.loads((KIT / "outputs/generation_info.json").read_text())
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
cards_match_cleaning = (generation["input_sha256"] == sha(SD / "cleaning/output/requests_cleaned.csv")
                        == cleaning_info["output_sha256"]
                        and generation["input_record"]["cleaning_version"] == cleaning_info["cleaning_version"])
assert cards_match_cleaning, "The recorded card-generation input doesn't match the current cleaned file"

# Check card-object links before filtering, so a wrong split can't silently include or exclude records
assert manifest["card_id"].notna().all() and manifest["card_id"].is_unique, "Manifest card IDs are missing or duplicated"
linked = objects.merge(manifest[["card_id", "split"]], on="card_id", how="left", suffixes=("_object", "_card"),
                       validate="many_to_one")
assert linked["split_card"].notna().all(), "An object belongs to an unknown card"
assert linked["split_object"].eq(linked["split_card"]).all(), "An object's split differs from its card"

dev_cards = manifest[manifest["split"].isin(DEV_SPLITS)].reset_index(drop=True)
dev_objects = objects[objects["split"].isin(DEV_SPLITS)].reset_index(drop=True)
assert dev_cards["split"].value_counts().to_dict() == {"train": 360, "validation": 120}, "Unexpected development split sizes"

# Fingerprints saved with the frozen settings: each development image (name and contents) and both label files
image_records = [{"file_name": f, "sha256": sha(KIT / "images" / f)} for f in sorted(dev_cards["file_name"])]
dev_images_sha256 = hashlib.sha256(json.dumps(image_records, sort_keys=True).encode()).hexdigest()
label_sha256 = {name: sha(KIT / name) for name in ["manifest.csv", "icon_instances.csv"]}

print(f"Data version {DATA_VERSION}; cards made from cleaning version {generation['input_record']['cleaning_version']}")
print(f"Development cards: {(dev_cards['split'] == 'train').sum()} training, {(dev_cards['split'] == 'validation').sum()} validation; "
      f"{len(dev_objects):,} drawn objects ({(dev_objects['role'] == 'kit_item').sum()} kit items)")

Data version 2; cards made from cleaning version 1.1
Development cards: 360 training, 120 validation; 1,240 drawn objects (958 kit items)


## A. How regions are matched to objects

The matching rules are fixed before any setting is compared, so every setting is scored the same way.

A region **overlaps** an object when their bounding boxes share a positive area. Boxes that only meet at an edge or corner do not overlap. This is a box-based association rule; it does not establish that the region's foreground pixels belong to that object. Even a small overlap counts under this rule.

### Region outcomes

Each region is classified as:

- **False region:** it overlaps no recorded object.
- **Merged region:** it overlaps two or more recorded objects.
- **Single-object region:** it overlaps exactly one recorded object.

Recorded objects include both kit items and background objects.

### Object outcomes

Each object receives one outcome, checked in this order:

1. **Missed:** no region overlaps it.
2. **Merged:** a merged region overlaps it. A merge never counts as a clean detection, even if its box fits the object.
3. **Split:** two or more regions overlap it.
4. **One region, good fit:** exactly one region overlaps it, that region overlaps no other object, and the boxes have **IoU ≥ 0.5**.
5. **One region, poor fit:** the same one-to-one match, but with IoU < 0.5.

**IoU**, or intersection over union, is the shared area of the two boxes divided by their total combined area, counting the shared area only once.

These outcomes are mutually exclusive. If an object is involved in both a merge and a split, it is reported as merged because that check comes first.

Only outcome 4 counts as a **clean detection**. It is one-to-one: the object has one region, and that region has one object, so two regions cannot both receive credit for the same object.

### Main measure

The main measure is the **share of kit items with a clean detection**. False regions, merged regions, and the other object outcomes are reported alongside it.

Clean detections should make classification easier, but notebook 03's end-to-end evaluation will show whether imperfect regions still lead to correct item-presence decisions. A classifier may recognise an item even when its crop fits poorly.

Objects missed during segmentation remain in the evaluation population; they are not excluded because no suitable crop was found. Notebook 03 evaluates the complete pipeline's item-presence decisions separately.

In [2]:
IOU_GOOD = 0.5


def iou(r, o):
    inter = max(0, min(r[2], o[2]) - max(r[0], o[0])) * max(0, min(r[3], o[3]) - max(r[1], o[1]))
    return inter / ((r[2] - r[0]) * (r[3] - r[1]) + (o[2] - o[0]) * (o[3] - o[1]) - inter)


def overlaps(r, o):
    # Positive shared area; boxes that only meet at an edge or corner do not overlap
    return r[0] < o[2] and r[2] > o[0] and r[1] < o[3] and r[3] > o[1]


def match_card(regions, card_objects):
    # Returns (region rows, object rows) for one card using the rules above
    boxes = [(o.x0, o.y0, o.x1, o.y1) for o in card_objects.itertuples()]
    overlapped_by_region = [[j for j, b in enumerate(boxes) if overlaps(r, b)] for r in regions]
    region_rows = []
    for k, (r, hits) in enumerate(zip(regions, overlapped_by_region)):
        kind = "false" if not hits else ("merged" if len(hits) > 1 else "single")
        region_rows.append({"region": k, "x0": r[0], "y0": r[1], "x1": r[2], "y1": r[3], "region_type": kind,
                            "objects_overlapped": len(hits), "object_index": hits[0] if kind == "single" else None})
    object_rows = []
    # object_index is the zero-based position within card_objects for this card (saved later together with card_id)
    for j, (o, b) in enumerate(zip(card_objects.itertuples(), boxes)):
        hits = [k for k, t in enumerate(overlapped_by_region) if j in t]
        best = max((iou(regions[k], b) for k in hits), default=0.0)
        if not hits:
            outcome = "missed"
        elif any(len(overlapped_by_region[k]) > 1 for k in hits):
            outcome = "merged"
        elif len(hits) > 1:
            outcome = "split"
        else:
            outcome = "one_region_good_fit" if best >= IOU_GOOD else "one_region_poor_fit"
        object_rows.append({"object_index": j, "outcome": outcome, "best_iou": best, "regions_overlapping": len(hits)})
    return region_rows, object_rows


# Self-test of the rules on hand-made boxes, so the matching logic is verified before it is used
_objs = pd.DataFrame({"x0": [10, 100, 200, 300], "y0": [10, 10, 10, 10], "x1": [50, 140, 240, 340], "y1": [50, 50, 50, 50]})
_regions = [(10, 10, 50, 50),          # object 0: good fit
            (95, 5, 245, 55),          # overlaps objects 1 and 2: merged
            (300, 10, 315, 50), (320, 10, 340, 50),   # object 3: split
            (400, 10, 420, 30)]        # false region
_r, _o = match_card(_regions, _objs)
assert [x["outcome"] for x in _o] == ["one_region_good_fit", "merged", "merged", "split"]
assert [x["region_type"] for x in _r] == ["single", "merged", "single", "single", "false"]
_, _missed = match_card([], _objs)                                  # no regions: every object is missed
assert all(o["outcome"] == "missed" for o in _missed)
_, _poor = match_card([(10, 10, 20, 20)], _objs.iloc[:1])           # small region over one object: poor fit
assert _poor[0]["outcome"] == "one_region_poor_fit"
assert not overlaps((0, 0, 10, 10), (10, 0, 20, 10))                # meeting at an edge is not an overlap
_, _edge = match_card([(10, 10, 30, 50)], _objs.iloc[:1])           # IoU exactly 0.5 counts as a good fit
assert _edge[0]["best_iou"] == 0.5 and _edge[0]["outcome"] == "one_region_good_fit"
print("Matching rules pass their self-test (good fit, poor fit, merged, split, missed, false region, edge contact, IoU = 0.5)")

Matching rules pass their self-test (good fit, poor fit, merged, split, missed, false region, edge contact, IoU = 0.5)


## B. Reconciling the pilot

The shared segmentation pilot (`shared/synthetic-data/checks/`) reported its best training setting as 568 objects found as one region with a good fit, 340 with a poor fit, 11 split and 1 missed. Those four groups add up to all 920 training objects. Its 65 merged regions were a **separate count of regions**, not objects: under the pilot's rule, an object counted as "found as one region" whenever exactly one region overlapped it, even if that region also covered another object.

This cell reruns the pilot's best setting on the same training cards and scores it twice: once with the pilot's original counting rule, to confirm the explanation above, and once with the new matching rules, to show how many of the pilot's "one region" objects were actually part of a merge. It also counts merged regions, so the two counting units (regions and objects) can be compared directly.

"Best" refers to the pilot's original scoring rule. This rerun assesses that setting under the stricter matching rules; it does not assume the setting will remain best.

In [3]:
PILOT_CONTENT = (slice(37, 318), slice(2, 478))   # the pilot's area: below the header, inside the 2-pixel border
GRAY_WEIGHTS = np.array([0.114, 0.587, 0.299], dtype=np.float32)   # OpenCV loads images as blue-green-red


def find_regions(diff, threshold, open_size, close_size, min_pixels, area, connectivity=8):
    mask = (diff > threshold).astype(np.uint8)
    if open_size:
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((open_size, open_size), np.uint8))
    if close_size:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size)))
    _, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=connectivity)
    x_off, y_off = area[1].start, area[0].start          # convert back to card coordinates
    return [(int(s[0]) + x_off, int(s[1]) + y_off, int(s[0] + s[2]) + x_off, int(s[1] + s[3]) + y_off)
            for s in stats[1:] if s[4] >= min_pixels]


def pilot_regions(path):
    # The pilot's best setting E: median 5, Gaussian background (sigma 25), colour distance > 20, open 3, close 9
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f"Could not open image: {path}")
    content = cv2.medianBlur(image, 5)[PILOT_CONTENT].astype(np.float32)
    diff = np.linalg.norm(content - cv2.GaussianBlur(content, (0, 0), 25), axis=2)
    return find_regions(diff, 20, 3, 9, 30, PILOT_CONTENT)


def pilot_rule(regions, card_objects):
    # The pilot's original rule: count the regions overlapping each object, ignoring whether a region also covers another object
    out = []
    for o in card_objects.itertuples():
        box = (o.x0, o.y0, o.x1, o.y1)
        hits = [r for r in regions if overlaps(r, box)]
        if not hits:
            out.append("missed")
        elif len(hits) > 1:
            out.append("split")
        else:
            out.append("one_region_good_fit" if iou(hits[0], box) >= IOU_GOOD else "one_region_poor_fit")
    return out


train_cards = dev_cards[dev_cards["split"] == "train"]
pilot_rows, old_rule, merged_regions = [], [], 0
for card in train_cards.itertuples():
    card_objects = dev_objects[dev_objects["card_id"] == card.card_id].reset_index(drop=True)
    found = pilot_regions(KIT / "images" / card.file_name)
    region_rows, obj_rows = match_card(found, card_objects)
    merged_regions += sum(r["region_type"] == "merged" for r in region_rows)
    pilot_rows.extend(obj_rows)
    old_rule.extend(pilot_rule(found, card_objects))

PILOT_OUTCOMES = ["one_region_good_fit", "one_region_poor_fit", "merged", "split", "missed"]
pilot = pd.DataFrame({
    "pilot's original rule": pd.Series(old_rule).value_counts(),
    "new matching rules": pd.Series([r["outcome"] for r in pilot_rows]).value_counts(),
}).reindex(PILOT_OUTCOMES).fillna(0).astype(int)
expected_objects = int((dev_objects["split"] == "train").sum())
assert len(pilot_rows) == len(old_rule) == expected_objects, "Not every training object was scored"
assert (pilot.sum() == expected_objects).all(), "Outcome counts don't add up to all training objects"
pilot.to_csv(OUTPUT_DIR / "pilot_reconciliation_train.csv")
print(f"Pilot setting E on {len(train_cards)} training cards, {expected_objects} objects:")
display(pilot)
print(f"Merged regions (each covering two or more objects): {merged_regions}")

Pilot setting E on 360 training cards, 920 objects:


,pilot's original rule,new matching rules
one_region_good_fit,568,568
one_region_poor_fit,340,201
merged,0,148
split,11,2
missed,1,1


Merged regions (each covering two or more objects): 65
